# MVT optimizer: 2023-24 FantasyGo contests

Haugh & Singal's mean-variance "beat the benchmark" optimizer (`src/mvt.py`) on the structural model's predictions.
Each gameweek it picks the FantasyGo team (11 players, budget 80, captain x2, vice-captain x1.5, 1 GK, 3-5 DEF,
2-5 MID, 1-3 FWD, at most 3 per club) that maximizes

$$\text{Prob}\{w^T\delta > G^{(r')}(\mathbb{W}_o, \delta)\}, \qquad \delta \sim N(\mu_\delta, \Sigma_\delta)$$

* $\mu_\delta$: `pred` from `structural.py` (trained with 2022-23 history), the expected points including the chance
  of not playing
* $\Sigma_\delta = \mathrm{diag}(\sigma_\delta)\,\rho_\delta\,\mathrm{diag}(\sigma_\delta)$: each player's sd of past errors
  (`total_points - pred`), and one correlation per (same team / opponents, position pair), pooled over earlier
  gameweeks (Appendix C)
* $\mathbb{W}_o$: the contest field, simulated with the team generator and the league selection model
* $G^{(r')}$: the score at rank `r_pct` of the field (0.8 = the top 20%)

By the mean-variance reduction the solution is $w(\lambda) \in \arg\max_w\, w^T\mu_\delta \mp \lambda\,(w^T\Sigma_\delta w -
2w^T\sigma_{\delta,G})$ for some $\lambda \ge 0$; $\lambda^*$ is the grid value whose team beats $G^{(r')}$ in the most Monte
Carlo draws. $\lambda = 0$ is the **base** optimizer: the highest expected points on the same shortlist.

Run the season with `cd src && python mvt.py` (about 2-3 minutes per gameweek; cached in `outputs/23-24/mvt/`, resumable).
This notebook reads the cache.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import mvt  # noqa: E402

s = mvt.Settings()  # season 23-24, predictions with 22-23 history, simulated field, r_pct 0.8
results, inp = mvt.run(s)  # from the cache; computes and caches any missing gameweek
print(f"{len(results)} gameweeks, GW{min(results)}-{max(results)}, beating the top {1 - s.r_pct:.0%} of the field")

## One gameweek: the frontier and the chosen team

In [ ]:
GW = 30
res = results[GW]
print(f"GW{GW}: G^(r') mean {res['G_mean']:.1f}, sd {res['G_sd']:.1f}; lambda* = {res['lambda_star']}")
res["frontier"].round(3)

In [ ]:
team = res["teams"][res["lambda_star"]]
print(f"cost {team['value'].sum():.1f}, expected {(team['pred'] * team['w']).sum():.1f}, "
      f"actual {mvt.team_points(team, GW, inp):.0f} points")
team[["pos", "name", "team", "value", "pred", "p_play", "captain", "vice"]].round(2)

## Every gameweek: the MVT team against the base team

Each team is scored on the actual points (captain x2, vice x1.5) and ranked in that gameweek's real FantasyGo contest:
rank = 1 + entries with more points; `prize` is what the entry at that rank won.

In [ ]:
evaluation = mvt.evaluate(results, inp, s)
display(mvt.summary(evaluation, s).round(3))
evaluation.round(2)

## Through the season

In [ ]:
SERIES = {"base": ("Base (lambda = 0)", "#2a78d6"), "mvt": ("MVT (lambda*)", "#eb6834")}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, (name, color) in SERIES.items():
    axes[0].plot(evaluation.index, evaluation[f"{label}_points"].cumsum(), color=color, lw=2, label=name)
    axes[1].plot(evaluation.index, 100 * evaluation[f"{label}_rank"] / evaluation["entries"], color=color, lw=2,
                 marker="o", ms=3, label=name)
axes[1].axhline(100 * (1 - s.r_pct), color="#999", lw=1, ls="--")
axes[1].text(evaluation.index.max(), 100 * (1 - s.r_pct), f" target: top {1 - s.r_pct:.0%}", va="bottom",
             ha="right", fontsize=9, color="#555")
axes[1].invert_yaxis()
axes[0].set(title="Cumulative actual points", xlabel="Gameweek", ylabel="Points")
axes[1].set(title="Rank in the FantasyGo contest", xlabel="Gameweek", ylabel="Rank (% of entries, top = 0)")
for ax in axes:
    ax.title.set(loc="left", fontsize=11, fontweight="bold")
    ax.grid(axis="y", color="#e5e5e5", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#bbb")
    ax.tick_params(colors="#555")
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

## How often MVT moved away from the base team

In [ ]:
lam = evaluation["lambda_star"].value_counts().sort_index().rename("gameweeks")
diff = evaluation[evaluation["lambda_star"] > 0]
print(f"lambda* > 0 in {len(diff)} of {len(evaluation)} gameweeks; there MVT scored "
      f"{diff['mvt_points'].sum():.0f} points against the base team's {diff['base_points'].sum():.0f}")
lam.to_frame()

## Is the simulated field like the real one?

The field comes from the team generator and the selection model (`selection.ipynb`). For every gameweek, 10 simulated
fields from each generator, one selection vector $p$ per team (`per_team_p`, the earlier version) and one $p$ for the
whole field (`shared_p`, the default, as in the selection model's likelihood), next to the real contest and the
previous gameweek's teams, all scored with the actual points:

* `cost`: mean team cost; `cut_points`: the points needed for the top `1 - r_pct`, the $G^{(r')}$ to beat
* `share_corr`: rank correlation of how often each player is picked, against the real field
* `top5_<pos>`: share of the position's picks that went to its five most-picked players (how concentrated the field is)
* `teams`, `accept_rate`: simulated teams and the share of attempts that met the budget and club rules (at most 20,000
  attempts per field); fields with fewer than 90% of the entries are listed, and a field with no team is left out

The last table asks whether the simulated fields cover the real cut: with 10 fields and a calibrated field model, the
real cut is below or above all ten in about 2/11 of the gameweeks. Cached in `outputs/23-24/mvt/` (about 25 minutes
to compute).

In [ ]:
fields = mvt.field_check(s)  # from the cache; simulates and caches any missing gameweek
means, short, coverage = mvt.field_tables(fields, s)
display(means.round(3))
print("Simulated fields with fewer than 90% of the entries:")
display(short.round(4))
coverage.round(2)

## Caveats

* $\widehat{\text{Prob}}$ uses the same $\mu_\delta$ for our team and the field; our team is chosen for high $\mu_\delta$, so
  any optimism in the predictions makes it optimistic.
* $\Sigma_\delta$ rests on the gameweeks before each one (5 at GW6), pooled per position early on; the eigenvalue floor
  keeps it usable but not accurate.
* The simulated field rests on the selection model and on the captain rule (two highest predictions).
* Only the `shortlist` best-predicted players per position are candidates; some large-$\lambda$ solves stop at the time
  limit with the best team found so far.
* In a double gameweek the predictions and actual points cover the player's first fixture only, while the real
  contest scores both.